# 09 - Adapting the workflow to a new sensor

This notebook is a template/checklist, not a full second pipeline -- the scientific
*decisions* (Steps 1, 4, 5, 6, 7 below) still require human judgment and are not automated.
But the code cells that load data, audit coverage, generate an artificial-gap pool, and score
baselines are real and executed: they run against `data_public/oxygen/`, the actual second
sensor at this site, using the exact same `src/coastal_gap_reconstruction/` functions as the
chlorophyll case study (called with `target_col="oxygen_mean_mgL"`,
`eligible_col="eligible_ge_18"` instead of the chlorophyll defaults -- see
`config/contracts/oxygen_target_contract.yaml`). This is the proof that the pipeline code
itself, not just the narrative, transfers to a second dataset. See
`notebooks/10_oxygen_case_study.ipynb` for the fuller oxygen results writeup, and
`config/contracts/target_contract_template.yaml` for a blank contract to fill in for a sensor
not already in this repository.

## Step 0: Confirm this repository's scope

Read `docs/methodology/target_and_gap_construction.md`,
`docs/methodology/validation_protocol.md`, and `docs/evidence_hierarchy.md`
before adapting anything. The target definition, eligibility rule, and
validation protocol are the load-bearing scientific decisions in this
workflow; copying the code without understanding these choices risks
silently producing invalid results for a new sensor.


## Step 1: Define the new target

For a new sensor (e.g. oxygen), you will need to make the same decisions
documented in `docs/methodology/target_and_gap_construction.md` for
chlorophyll:

- What is the daily aggregation rule (mean? median?) and why?
- What hourly-validity threshold defines an "eligible" day?
- What is the full valid range, and how are invalid/negative/out-of-range
  raw readings handled?
- Are there known sensor drift, calibration, or fouling issues specific to
  this variable that need a different QA approach than chlorophyll?

These are scientific decisions, not implementation details -- they should
be made deliberately and documented, not inherited by default from the
chlorophyll target's design choices.


In [1]:
# Live example: oxygen is the real second sensor this checklist produced. This cell
# actually loads it (no placeholder path) -- see config/contracts/oxygen_target_contract.yaml
# for the column names used below.

import sys
sys.path.insert(0, "../src")
from coastal_gap_reconstruction.data_loading import load_daily_target

OXYGEN_TARGET_COL = "oxygen_mean_mgL"
OXYGEN_ELIGIBLE_COL = "eligible_ge_18"

oxygen_df = load_daily_target("../data_public/oxygen/oxygen_daily_target.csv")
oxygen_df[[OXYGEN_TARGET_COL, OXYGEN_ELIGIBLE_COL]].head()

,oxygen_mean_mgL,eligible_ge_18
date,,
2015-07-01,NaN,False
2015-07-02,NaN,False
2015-07-03,NaN,False
2015-07-04,NaN,False
2015-07-05,NaN,False


## Step 2: Re-run the gap audit

Use `notebooks/01_target_and_gap_audit.ipynb` as a template: regenerate
coverage statistics, real-gap inventory, and eligible-run structure for the
new target. Missingness patterns may differ substantially between sensors
(e.g. oxygen sensors may have different fouling/drift failure modes than
chlorophyll fluorometers).


In [2]:
from coastal_gap_reconstruction.gap_detection import coverage_summary, find_real_gaps

coverage_summary(oxygen_df, eligible_col=OXYGEN_ELIGIBLE_COL)

{'n_days_total': 3988,
 'n_days_eligible': 2880,
 'eligible_fraction': 0.7221664994984955,
 'n_real_gaps': 125,
 'longest_real_gap_days': 256}

## Step 3: Reconstruct the artificial-gap validation pool

Use `src/coastal_gap_reconstruction/artificial_gap_validation.py`'s
`generate_gap_candidates` function against the new target's eligible-run
structure. Decide whether the same gap lengths (1, 3, 7, 14, 30, 45, 60
days) remain appropriate, or whether the new sensor's missingness pattern
calls for a different set.


In [3]:
from coastal_gap_reconstruction.artificial_gap_validation import generate_gap_candidates

oxygen_gap_pool = generate_gap_candidates(
    oxygen_df,
    gap_lengths=[1, 3, 7, 14, 30],
    target_col=OXYGEN_TARGET_COL,
    eligible_col=OXYGEN_ELIGIBLE_COL,
)
oxygen_gap_pool["gap_length"].value_counts().sort_index()

gap_length
1     100
3     100
7     100
14    100
30     48
Name: count, dtype: int64

## Step 4: Re-evaluate baselines first

Run `notebooks/03_baselines.ipynb`'s logic against the new target before
trying anything more sophisticated. The relative ranking of climatology
vs. persistence vs. interpolation may differ for a variable with different
seasonal/autocorrelation structure than chlorophyll.


In [4]:
from coastal_gap_reconstruction.artificial_gap_validation import apply_artificial_gap
from coastal_gap_reconstruction.baseline_imputation import run_all_baselines
from coastal_gap_reconstruction.scoring_metrics import compute_gap_metrics
import pandas as pd

sample_gap = oxygen_gap_pool.iloc[0]
masked = apply_artificial_gap(
    oxygen_df, sample_gap["start_date"], int(sample_gap["gap_length"]), target_col=OXYGEN_TARGET_COL
)
preds = run_all_baselines(
    masked, sample_gap["start_date"], int(sample_gap["gap_length"]),
    target_col=OXYGEN_TARGET_COL, eligible_col=OXYGEN_ELIGIBLE_COL,
)
rows = compute_gap_metrics(
    oxygen_df, preds, sample_gap["start_date"], int(sample_gap["gap_length"]),
    sample_gap["gap_id"], sample_gap.to_dict(), target_col=OXYGEN_TARGET_COL,
)
pd.DataFrame(rows)[["method", "mae", "rmse", "bias", "coverage"]]

,method,mae,rmse,bias,coverage
0,clim_monthly,0.7588,0.7588,-0.7588,1.0
1,persistence,1.1081,1.1081,1.1081,1.0
2,linear_interp,0.2849,0.2849,0.2849,1.0


## Step 5: Re-select predictor features

The curated chlorophyll feature table includes chlorophyll-specific
covariates (a satellite chlorophyll proxy, upwelling indices tuned for
biological productivity). For a new sensor, re-evaluate which external
predictors are physically relevant -- do not assume the same feature table
transfers without justification.


## Step 6: Re-run engineered tabular / TS-ICL methods

Once a baseline floor and a relevant feature set exist for the new sensor,
revisit `docs/methodology/model_families.md` and
`notebooks/06_tsicl_zero_shot_imputation.ipynb` to apply the same model
ladder. Re-check whether a satellite proxy covariate is meaningful for the
new variable -- for oxygen, for example, there is no obvious "satellite
oxygen proxy" analogous to satellite chlorophyll, so the leading TS-ICL
configuration for chlorophyll may not transfer directly.


## Step 7: Re-establish the evidence hierarchy

Apply the same discipline described in `docs/evidence_hierarchy.md`:
artificial-gap validation results are the only validation-grade evidence;
real-gap candidate outputs are plausibility only. This discipline does not
change across sensors.
